# Inventory Forecast Agent with MCP Server

This notebook demonstrates how to invoke the **inventory-forecast-endpoint** using an AI agent connected via the **Model Context Protocol (MCP)**.

| Step | Description |
|------|-------------|
| 1 | **Install packages** — `databricks-langchain`, `mcp`, `fastmcp` |
| 2 | **Configuration** — Endpoint name, workspace URL, LLM endpoint |
| 3 | **Define MCP Server** — FastMCP server exposing the forecast endpoint as a tool |
| 4 | **Build Agent** — LangGraph ReAct agent connected to the MCP server |
| 5 | **Test Agent** — Invoke forecasts via natural language queries |

**Architecture:**
```
User (natural language) → LangGraph Agent → MCP Server → Model Serving Endpoint → Predictions
```

In [0]:
%pip install databricks-langchain langgraph fastmcp mcp -q
dbutils.library.restartPython()

In [0]:
# -- Configuration -----------------------------------------------------------------------

# Model Serving endpoint deployed in the previous notebook
ENDPOINT_NAME = "inventory-forecast-endpoint"

# LLM endpoint for the agent (foundation model on Databricks)
LLM_ENDPOINT = "databricks-claude-sonnet-4-5"

# Workspace details
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
WORKSPACE_URL = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
TOKEN = notebook_context.apiToken().get()

print(f"Serving endpoint : {ENDPOINT_NAME}")
print(f"LLM endpoint     : {LLM_ENDPOINT}")
print(f"Workspace URL    : {WORKSPACE_URL}")

In [0]:
import os
import tempfile

# Write the MCP server script to a file
# The server exposes the forecast endpoint as an MCP tool
mcp_server_code = f'''
import json
import requests
import pandas as pd
from datetime import datetime, timedelta
from fastmcp import FastMCP

mcp = FastMCP("Inventory Forecast Server")

ENDPOINT_URL = "{WORKSPACE_URL}/serving-endpoints/{ENDPOINT_NAME}/invocations"
TOKEN = "{TOKEN}"

@mcp.tool()
def forecast_inventory(product_id: str, days: int = 5, start_date: str = "2024-07-01") -> str:
    """
    Forecast inventory levels for a specific product (SKU).
    
    Args:
        product_id: The product SKU identifier (e.g., "SKU-001", "SKU-002", "SKU-003", "SKU-004", "SKU-005")
        days: Number of days to forecast (1-30, default 5)
        start_date: Start date for forecast in YYYY-MM-DD format (default "2024-07-01")
    
    Returns:
        JSON string with forecasted inventory levels for each day
    """
    days = min(max(1, days), 30)
    
    start = pd.Timestamp(start_date)
    records = [
        {{"date": str(pd.Timestamp(d).date()), "product_id": product_id}}
        for d in pd.date_range(start, periods=days, freq="D")
    ]
    
    payload = {{"dataframe_records": records}}
    headers = {{
        "Authorization": f"Bearer {{TOKEN}}",
        "Content-Type": "application/json",
    }}
    
    try:
        resp = requests.post(ENDPOINT_URL, headers=headers, json=payload)
        if resp.status_code == 200:
            result = resp.json()
            predictions = result.get("predictions", result)
            
            forecast_results = []
            if isinstance(predictions, list):
                for i, pred in enumerate(predictions):
                    date_str = records[i]["date"]
                    value = pred if isinstance(pred, (int, float)) else pred.get("yhat", pred)
                    forecast_results.append({{
                        "date": date_str,
                        "product_id": product_id,
                        "predicted_inventory_level": round(float(value), 2)
                    }})
            else:
                forecast_results = [{{"raw_response": str(predictions)}}]
            
            return json.dumps({{
                "status": "success",
                "product_id": product_id,
                "forecast_days": days,
                "start_date": start_date,
                "forecasts": forecast_results
            }}, indent=2)
        else:
            return json.dumps({{
                "status": "error",
                "code": resp.status_code,
                "message": resp.text[:500]
            }})
    except Exception as e:
        return json.dumps({{"status": "error", "message": str(e)}})


@mcp.tool()
def list_available_products() -> str:
    """
    List all available product SKUs that can be forecasted.
    
    Returns:
        JSON string with available product IDs and descriptions
    """
    products = [
        {{"product_id": "SKU-001", "description": "Product 1 - Base level ~900"}},
        {{"product_id": "SKU-002", "description": "Product 2 - Base level ~1100"}},
        {{"product_id": "SKU-003", "description": "Product 3 - Base level ~1200"}},
        {{"product_id": "SKU-004", "description": "Product 4 - Base level ~1000"}},
        {{"product_id": "SKU-005", "description": "Product 5 - Base level ~1300"}},
    ]
    return json.dumps({{
        "available_products": products,
        "note": "Forecasts are available for dates starting from 2024-07-01 (up to 30 days)"
    }}, indent=2)


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

mcp_server_path = os.path.join(tempfile.gettempdir(), "inventory_mcp_server.py")
with open(mcp_server_path, "w") as f:
    f.write(mcp_server_code)

print(f"MCP Server script written to: {mcp_server_path}")
print(f"\nExposed tools:")
print(f"  1. forecast_inventory(product_id, days, start_date)")
print(f"  2. list_available_products()")

In [0]:
import asyncio
import sys
import nest_asyncio
nest_asyncio.apply()
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def get_mcp_tools():
    """Connect to the MCP server via stdio and retrieve available tools."""
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[mcp_server_path],
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            print(f"MCP Server connected! Found {len(tools_result.tools)} tools:\n")
            for t in tools_result.tools:
                print(f"  Tool: {t.name}")
                print(f"  Desc: {t.description[:100]}")
                print()
            return tools_result.tools, server_params

mcp_tools_info, server_params = asyncio.run(get_mcp_tools())
print("MCP tools loaded successfully!")

In [0]:
from langchain_core.tools import StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from databricks_langchain import ChatDatabricks
import json

# -- MCP tool wrapper functions ------------------------------------------------
# These call the MCP server via stdio transport each time

async def _call_mcp_tool(tool_name: str, arguments: dict) -> str:
    """Call a tool on the MCP server via stdio."""
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            return result.content[0].text if result.content else "No response"


def forecast_inventory(product_id: str, days: int = 5, start_date: str = "2024-07-01") -> str:
    """Forecast inventory levels for a specific product SKU.
    
    Args:
        product_id: The product SKU (e.g., SKU-001 through SKU-005)
        days: Number of days to forecast (1-30)
        start_date: Start date in YYYY-MM-DD format
    """
    return asyncio.run(_call_mcp_tool("forecast_inventory", {
        "product_id": product_id,
        "days": days,
        "start_date": start_date,
    }))


def list_available_products() -> str:
    """List all available product SKUs that can be forecasted."""
    return asyncio.run(_call_mcp_tool("list_available_products", {}))


# -- Create LangChain tools from MCP wrappers ----------------------------------
langchain_tools = [
    StructuredTool.from_function(
        func=forecast_inventory,
        name="forecast_inventory",
        description="Forecast inventory levels for a specific product SKU. Available products: SKU-001 to SKU-005. Dates start from 2024-07-01, up to 30 days.",
    ),
    StructuredTool.from_function(
        func=list_available_products,
        name="list_available_products",
        description="List all available product SKUs that can be forecasted.",
    ),
]

# -- Build the ReAct agent -----------------------------------------------------
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0)

SYSTEM_PROMPT = """You are an inventory forecasting assistant. You help users get predictions 
for stock inventory levels using the forecast tools available. 

When providing forecasts:
- Always show dates and predicted inventory levels clearly
- Summarize trends (increasing, decreasing, stable)
- Mention any notable patterns
- Use the available tools to get actual predictions from the model"""


class SimpleReActAgent:
    """Lightweight ReAct agent using tool-calling LLM."""
    def __init__(self, llm, tools, system_prompt):
        self.llm = llm.bind_tools(tools)
        self.tools = {t.name: t for t in tools}
        self.system_prompt = system_prompt

    def invoke(self, input_dict):
        messages = [SystemMessage(content=self.system_prompt)]
        for msg in input_dict["messages"]:
            if isinstance(msg, dict) and msg.get("role") == "user":
                messages.append(HumanMessage(content=msg["content"]))
            else:
                messages.append(msg)

        for _ in range(10):  # max iterations
            response = self.llm.invoke(messages)
            messages.append(response)

            if not response.tool_calls:
                break

            for tc in response.tool_calls:
                tool = self.tools[tc["name"]]
                result = tool.invoke(tc["args"])
                messages.append(ToolMessage(content=result, tool_call_id=tc["id"]))

        return {"messages": messages}


agent = SimpleReActAgent(llm, langchain_tools, SYSTEM_PROMPT)

print("ReAct Agent created!")
print(f"  LLM     : {LLM_ENDPOINT}")
print(f"  Tools   : {[t.name for t in langchain_tools]}")
print(f"  Backend : MCP Server -> Model Serving Endpoint")

In [0]:
# -- Test the agent with natural language queries --------------------------------

test_queries = [
    "What products can you forecast?",
    "Give me a 7-day inventory forecast for SKU-001 starting July 1st 2024",
    "Compare the 5-day forecast for SKU-002 and SKU-005 starting from 2024-07-10",
]

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f" Query {i}: {query}")
    print(f"{'='*70}\n")
    
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    
    # Print the agent's final response
    final_message = result["messages"][-1]
    print(f"Agent Response:\n{final_message.content}")
    print()

In [0]:
# -- Interactive: Change the query and re-run this cell --------------------------

user_query = "What's the predicted inventory for SKU-003 over the next 10 days starting July 15th?"

print(f"Query: {user_query}\n")
result = agent.invoke({"messages": [{"role": "user", "content": user_query}]})
print(f"Agent:\n{result['messages'][-1].content}")